

`Pweak => P` by type_props  is the type checking principle.



In [ ]:
from dataclasses import dataclass, replace, field
from typing import Optional, Callable
import pprint
import subprocess
import eprover
import functools



@dataclass(frozen=True, slots=True)
class Decl:
    name: str
    arity: int
    is_var: bool = False

    def __post_init__(self):
        if self.is_var and not self.name[0].isupper():  # or self.name[0] == "_"):
            raise ValueError("Variable not capitalized", self)
        elif not self.is_var and self.name[0].isupper():
            raise ValueError("Constant cannot be capitalized")

    def __call__(self, *args: "Term") -> "Term":
        assert (
            len(args) == self.arity
        ), f"Expected {self.arity} arguments, got {len(args)}"
        return Term(self, args)

class PDecl(Decl):
    def __call__(self, *args: "Term") -> "Term":
        assert (
            len(args) == self.arity
        ), f"Expected {self.arity} arguments, got {len(args)}"
        return PTerm(self, args)

LT = PDecl("lt", 2)
GT = PDecl("gt", 2)

@dataclass(frozen=True, slots=True)
class Term:
    decl: Decl
    args: tuple["Term", ...]

    def __post_init__(self):
        assert all(isinstance(a, Term) for a in self.args)

    def __lt__(self, other: "Term") -> "Prop":
        return LT(self, other)
    def __gt__(self, other: "Term") -> "Prop":
        return GT(self, other)
    def __eq__(self, other: "Term") -> "Prop":
        return Eq(self, other)

    def eq(self, other: "Term") -> bool:
        return self.decl == other.decl and all(
            s.eq(o) for s, o in zip(self.args, other.args)
        )


    def __call__(self, *args):
        assert len(args) > 0
        if len(args) == 1:
            return app(self, args[0])
        else:
            return app(self, args[0])(*args[1:])

    def __str__(self) -> str:
        return self.tptp()

    def tptp(self) -> str:
        if self.decl.arity == 0:
            return self.decl.name
        else: 
            return f"{self.decl.name}({', '.join(arg.tptp() for arg in self.args)})"

    def fvs(self) -> set["Term"]:
        # free variables in the expression
        if self.decl.is_var:
            return {self}
        else:
            return set().union(*(arg.fvs() for arg in self.args))

    def decls(self) -> set[Decl]:
        decls = set()
        todo = [self]
        while todo:
            t = todo.pop()
            decl = t.decl
            if decl.is_quantifier:
                todo.append(t.args[1])
            else:
                decls.add(decl)
                todo.extend(t.args)
        return decls


@dataclass(frozen=True, slots=True)
class And: left: "Prop"; right: "Prop"
@dataclass(frozen=True, slots=True)
class Or: left: "Prop"; right: "Prop"
@dataclass(frozen=True, slots=True)
class Not: arg: "Prop"
@dataclass(frozen=True, slots=True)
class Imp:  hyp: "Prop"; conc: "Prop"
@dataclass(frozen=True, slots=True)
class Eq: left: "Term"; right: "Term"
@dataclass(frozen=True, slots=True)
class NEq: left: "Term"; right: "Term"
@dataclass(frozen=True, slots=True)
class Iff: left: "Prop"; right: "Prop"
@dataclass(frozen=True, slots=True)
class Quantifier:
    vs : tuple["Term", ...]
    bounds: tuple["Prop", ...]
    body: "Prop"
class ForAll(Quantifier): ...
class Exists(Quantifier): ...

class PTerm(Term): ...

type Prop = And | Or | Not | Imp | ForAll | Exists | Eq | NEq | PTerm | Iff | bool

has_type = PDecl("has_type", 2)
app = Decl("app", 2)  # app. Sometimes PDecl? Call / call

    
def Const(name: str) -> Term:
    decl = Decl(name, 0)
    t = decl()
    return t
def Consts(names: str) -> list[Term]:
    return [Const(name) for name in names.split()]

@dataclass(frozen=True, slots=True)
class Proof:
    thm: Prop
    reasons: list[object]

    def __repr__(self) -> str:
        return "|- " + tptp(self.thm)

def fvs(p : Prop) -> set["Term"]:
    if isinstance(p, bool):
        return set()
    if isinstance(p, Term):
        return p.fvs()
    match p:
        case Not(a):
            return fvs(a)
        case And(a, b) | Or(a, b) | Imp(a, b) | Iff(a, b) | Eq(a, b) | NEq(a, b):
            return fvs(a) | fvs(b)
        case ForAll(vs, bounds, body) | Exists(vs, bounds, body):
            bound_fvs = set().union(*(fvs(b) for b in bounds))
            body_fvs = fvs(body)
            return (body_fvs - set(vs)) | bound_fvs
    raise TypeError(p)

def forallize(*p : Prop) -> Prop:
    if len(p) == 0:
        raise ValueError("Expected at least one proposition")
    elif len(p) == 1:
        p = p[0]
    elif len(p) == 2:
        p = Imp(p[0], p[1])
    else:
        p = Imp(functools.reduce(And, p[:-1]), p[-1])
    vs = fvs(p)
    if len(vs) != 0:
        return ForAll(
            vs=tuple(vs),
            bounds=tuple(),
            body=p
        )
    return p

def axiom(*p: Prop) -> Proof:
    # Ya need axioms. What can I say
    #assert (
    #    len(p.fvs()) == 0
    #), f"Expected closed term, got {p}, free variables: {p.fvs()}"
    #assert isinstance(p, Prop)
    return Proof(forallize(*p), reasons=["axiom"])

def tptp(p: Prop) -> str:
    if isinstance(p, bool):
        return "$true" if p else "$false"
    if isinstance(p, Term):
        return p.tptp()

    match p:
        case Not(a):
            return f"~({tptp(a)})"
        case And(a, b) | Or(a, b) | Imp(a, b) | Iff(a, b) | Eq(a, b) | NEq(a, b):
            op = {And: "&", Or: "|", Imp: "=>", Iff: "<=>",
                  Eq: "=", NEq: "!="}[type(p)]
            return f"({tptp(a)} {op} {tptp(b)})"
        case ForAll(vs, bounds, body) | Exists(vs, bounds, body):
            guard = " & ".join(tptp(b) for b in bounds)
            body = tptp(body)
            if guard:
                body = f"(({guard}) {' =>' if isinstance(p, ForAll) else ' &'} {body})"
            q = "!" if isinstance(p, ForAll) else "?"
            return f"{q}[{', '.join(tptp(v) for v in vs)}]: ({body})"
    raise TypeError(p)


def prove(*p: Prop, by=()) -> Proof:
    assert all(isinstance(pf, Proof) for pf in by)
    p = forallize(*p)
    lines = [f"fof(ax{n}, axiom, {tptp(pf.thm)})." for n, pf in enumerate(by)]
    lines.append(f"fof(goal, conjecture, {tptp(p)}).")
    res = subprocess.run(
        [eprover.eprover_path, "--cpu-limit=1"],
        input="\n".join(lines), text=True, capture_output=True,
    )
    if "SZS status Theorem" not in res.stdout:
        raise ValueError("Failed to prove", tptp(p), res.stdout, res.stderr)
    return Proof(p, reasons=["prove", *by])


axiom(True)
prove(True)
#prove(False)

type Type = Term

decl_types : dict[Decl, tuple[tuple[Type, ...], Proof]]= {}
const_types : dict[Term, tuple[Type, Proof]] = {}
nat = Const("nat")
bool_ = Const("bool")
unit = Const("unit")
arr = Decl("arr", 2)

true = Const("true")
false = Const("false")
ite = Decl("ite", 3)

zero = Const("zero")
succ = Decl("succ", 1)
pred = Decl("pred", 1)
iszero = Decl("iszero", 1)

def Var(name: str) -> Term:
    return Decl(name, 0, is_var=True)()
def Vars(names: str) -> list[Term]:
    return [Var(name) for name in names.split()]
X,Y,Z,T,T1,T2,T3 = Vars("X Y Z T T1 T2 T3")
# evaluation semantics vs equational.
ite_true = axiom(ite(true,T1,T2) > T1)
ite_false = axiom(ite(false,T1,T2) > T2)
#ite_true = axiom(ite(true,T1,T2) == T1)
true_type = axiom(has_type(true, bool_))
false_type = axiom(has_type(false, bool_))
zero_type = axiom(has_type(zero, nat))

succ_type = axiom(has_type(X, nat),
                  #------------------------
                   has_type(succ(X), nat))

iszero_type = axiom(has_type(X, nat),
                     #------------------------
                     has_type(iszero(X), bool_))

ite_type = axiom(has_type(T1, bool_), has_type(T2, T), has_type(T3, T), 
                 #--------------------------------
                 has_type(ite(T1,T2,T3), T))
#ite_false = axiom(ite(false,T1,T2) == T2)
#ite_cong1 = 

prove(has_type(ite(true, false, true), bool_), by=[ite_type,true_type,false_type])

# x : bool |- ite(x, false, true) : bool
prove(has_type(X, bool_), 
      has_type(ite(X, false, true), bool_), 
      by=[ite_type,true_type,false_type])





|- ![X]: ((has_type(X, bool) => has_type(ite(X, false, true), bool)))

https://www.cantab.net/users/antoni.diller/brackets/intro.html
https://okmij.org/ftp/tagless-final/ski.pdf

In [ ]:
s = Const("s")
k = Const("k")
i = Const("i")
s_eval = axiom(s(X,Y,Z) == X(Z)(Y(Z)))
k_eval = axiom(k(X,Y) == X)
i_eval = axiom(i(X) == X)
i_eval



|- ![X]: ((app(i, X) = X))

Hmm. So I can't do this unless I go all in on app style? At which point, maybe I should use thf.

forall T
pi() bracket asbstraction? Hmm.

In [51]:
from itertools import count

_fresh = count()
s, k, i = (Const(c) for c in "ski")

def bracket(x, t):
    if t.eq(x):
        return i
    if t.decl == app and t.args[1].eq(x) and x not in t.args[0].fvs():
        return t.args[0]                 # [x].(f x) = f
    if x not in t.fvs():
        return k(t)
    if t.decl == app:
        a, b = t.args
        return s(bracket(x, a), bracket(x, b))
    raise ValueError("variable occurs under a non-app symbol", t)

def lam(f):
    x = Decl(f"X{next(_fresh)}", 0, is_var=True)()
    body = f(x)
    if not isinstance(body, Term):       # another Python lambda
        body = lam(body)
    return bracket(x, body)

lam(lambda x: lam(lambda y: x(y)))

Term(decl=Decl(name='i', arity=0, is_var=False), args=())

In [ ]:
@dataclass(frozen=True, slots=True)
class Const:
    name: str
    def __post_init__(self):
        if not self.name.islower():
            raise ValueError("Constant name must be lowercase")
# no arity is still a problem?
@dataclass
class Var:
    name: str
    def __post_init__(self):
        if not self.name.isupper():
            raise ValueError("Variable name must be uppercase")
# And Or Implies
class Lambda:
    vs : list[Var]
    body: Term
class App:
    func: Term
    arg: Term
class Binder: # ($i > $i) >
    name : str
    arg : Lambda
type Term = Const | Var | Lambda | App
def thf(term: Term) -> str:
    match term:
        case Const(name):
            return name
        case Var(name):
            return name
        case Lambda(vs, body):
            vs_str = ",".join(f"{v.name} : $i" for v in vs)
            return f"^[{vs_str}]: {thf(body)}"
        case App(func, arg):
            return f"({thf(func)} @ {thf(arg)})"
def consts(t : Term) -> set[Const]:
    match t:
        case Const(name):
            return {t}
        case Var(name):
            return set()
        case Lambda(vs, body):
            return consts(body)
        case App(func, arg):
            return consts(func) | consts(arg)
def fvs(t: Term) -> set[Var]:
    match t:
        case Const(_):
            return set()
        case Var(_):
            return {t}
        case Lambda(vs, body):
            return fvs(body) - set(vs)
        case App(func, arg):
            return fvs(func) | fvs(arg)

type Prop = Const | Var | Lambda | App




In [ ]:

type Type = Term

decl_types : dict[Decl, tuple[tuple[Type, ...], Proof]]= {}
const_types : dict[Term, tuple[Type, Proof]] = {}
Nat = Const("nat")
Bool = Const("bool")
Unit = Const("unit")
Arr = Decl("arr", 2)

true = Const("true")
false = Const("false")

is_value = Decl("is_value", 1)
is_value = define("is_value", [x], x == true | x == false)
# hmm. Is value means we can't use =.

ev = define("ev", [t1,t2], 
            Or(
                t1 == true,
            )
            
            )



def type_decl(decl: Decl, *sorts: Type) -> None:
    assert len(sorts) >= 1
    decl_types[decl] = (sorts[:-1], sorts[-1], axiom(ForAll([Var("X" + str(i), bound=bound) for i, bound in enumerate(sorts[:-1])], sorts[-1](decl(*[Var("X" + str(i)) for i in range(len(sorts) - 1)])))))

# chapter 8.
zero = Const("zero", nat)
succ = Function("succ", nat, nat)
pred = Function("pred", nat, nat)
iszero = Function("iszero", nat, Bool)
ite = Decl("ite", 3)
ite_type = axiom(ForAll([t1,t2,t3,T], has_type(t1, Bool), has_type(t2, T), has_type(t3, T), has_type(ite(t1, t2, t3), T)))

# inversion cannot be proved if merely typing relation is considered an open concept
# We cannot prove metatheory. We can prove indicitual type terms though
prove(has_type(ite(iszero(zero), zero, pred(zero)), nat), by=type_theorems)



s = Const("s")
k = Const("k")
i = Const("i")

# abstraction directly kiselyov style?





def And(*args: PTerm) -> "PTerm":
    if len(args) == 0:
        return true
    elif len(args) == 1:
        return args[0]
    else:
        return Term(And0, (args[0], And(*args[1:])))






def Pred(name: str, arity: int) -> PDecl:
    return PDecl(name, arity)


def Function(name: str, *sorts) -> Decl:
    assert len(sorts) >= 2
    global contracts
    f = Decl(name, len(sorts) - 1)
    args = [Var("X" + str(i), bound=bound) for i, bound in enumerate(sorts[:-1])]
    contracts[f] = axiom(ForAll(args, sorts[-1](f(*args))))
    return f


true = PDecl("$true", 0)()
false = PDecl("$false", 0)()


def Var(name: str, bound=None) -> Term:
    return Term(Decl(name, 0, is_var=True, bound=bound), ())


def Vars(names: str, bound=None) -> list[Term]:
    return [Var(name, bound) for name in names.split()]








So the story goes, set theoy is axioms on top of first order logic, but type theory is 

When I speak of a "theory" or "logic", I am somewhat referring to operational reasoning mechanisms. The existence of tools like z3 / eprover / vampire is the point. I do not really always understand

But it is well known, you can use combinators to encode the lambda calculus. In that sense, you can talk about the lambda calculus without the syntactic or operational feature of binders or built-in beta substitution.




Superposition provers aren't as brute force as resolution/paramodulation. The term ordering allows some equations to be considered as rewrite rules rather than brute force explored equational reasoning.
The design of a good theory well adapted to a superposition prover would take the choice of term ordering and what it can order into consideration

What is a theory? It was kind of mind blowing to me that in some contexts, the definition of a theory is a set of formulas.


Do we need intuitionistic FOL?




|  Type Theory  |  FOL   |
|----|----|
| computation rules | unguard equations + term orderings |
| definitional type equality | =  unguarded equality |
| judgemental equality  G |- x = y : A |  has_type(x,A), has_type(y,A) -> x = y. guarded equality |
| prop equality          | eq(A, x, y) |
| proof terms            | proof(pf, form)  proof tracing. proof = has_type? Sure. |
| proof irrelevance   | proof(pf1, P), proof(pf2, P) -> pf2 = pf1 | Nice for compaction


The G is kind of tough. A shallow usage of G is to insert things deep inside of formulas.



has_type(A, type), has_type(B, type) -> has_type(arr(A,B), type)

